In [1]:
import os, random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# spike coordinates
SPIKE_START_1BASED = 21563
SPIKE_END_1BASED   = 25384
SPIKE_SLICE = slice(SPIKE_START_1BASED - 1, SPIKE_END_1BASED)

ATCG = set("acgt")
GAP_CHARS = set(["i", "-"])
LOSS_ON = GAP_CHARS  # only calculate loss in gap position


device: cuda


In [2]:
# === Load training sequences + load the voted scaffold, then extract Spike region gap indices ===
# This cell:
# 1) Reads the training sequences CSV (expects columns: ID, Sequence).
# 2) Normalizes ID/Sequence types and sequence formatting (strip + lowercase).
# 3) Loads the voted scaffold text file (concatenates non-empty lines).
# 4) Slices the scaffold to the Spike region using SPIKE_SLICE.
# 5) Computes L = spike scaffold length and finds all gap positions (indices) where char is in GAP_CHARS.


TRAIN_SEQ_CSV = "train_sequences.csv"
VOTE_SCAFFOLD_TXT = "vote_scaffold.txt"

# Load all training sequences
df_all = pd.read_csv(TRAIN_SEQ_CSV)     #read CSV into df_all
assert "ID" in df_all.columns and "Sequence" in df_all.columns  #Must have ID and sequence column!

# Ensure consistent types and formatting
df_all["ID"] = df_all["ID"].astype(str)         #Force ID into string
df_all["Sequence"] = df_all["Sequence"].astype(str).str.strip().str.lower()     #Force sequence into string, remove whitespace and empty lines, lowercase letters
print("loaded sequences:", len(df_all))

# Load the full voted scaffold (join all non-empty lines into one continuous string)
'''
"".join: Combine all valid lines into a single continuous string.
In python, .join is recommended command to splice strings.
'''
with open(VOTE_SCAFFOLD_TXT, "r") as f:
    vote_scaffold_full = "".join(line.strip() for line in f if line.strip()).lower()


# Extract Spike region scaffold and compute its length
vote_scaffold_spike = vote_scaffold_full[SPIKE_SLICE]
L = len(vote_scaffold_spike)
print("vote scaffold spike length:", L)

# Identify indices of all gap characters in the Spike scaffold (e.g., '-' or 'i')
G_idx = [i for i,ch in enumerate(vote_scaffold_spike) if ch in GAP_CHARS]       #traverse all positions and its index. If it belongs GAP_CHARS (i,-), collect it in G_idx
print("gap sites in vote scaffold spike:", len(G_idx))


loaded sequences: 6375
vote scaffold spike length: 3822
gap sites in vote scaffold spike: 199


In [3]:
# === Exclude chosen evaluation sequences from the training set ===
# This cell:
# 1) Defines a fixed list of sequence IDs selected as ground-truth evaluation samples.
# 2) Removes these sequences from the full dataset to form the training dataset.
# 3) Verifies dataset sizes and checks that none of the chosen IDs remain in training data.

CHOSEN_IDS = [
    "EPI_ISL_11935954",
    "EPI_ISL_12957585",
    "EPI_ISL_10094228",
    "EPI_ISL_12162837",
    "EPI_ISL_9494671",
]

# Ensure ID column is string-typed for safe comparison
df_all["ID"] = df_all["ID"].astype(str)

# Build training set by excluding the chosen evaluation sequences
df_train = df_all[~df_all["ID"].isin(CHOSEN_IDS)].reset_index(drop=True)

print("df_all size:", len(df_all))
print("df_train size (excluded 5 truths):", len(df_train))
print("still in train (should be empty):", [x for x in CHOSEN_IDS if x in set(df_train["ID"])])


df_all size: 6375
df_train size (excluded 5 truths): 6370
still in train (should be empty): []


In [4]:
# === Define nucleotide vocabulary and sequence encoding utility ===
# This cell:
# 1) Defines the fixed character vocabulary used by the model (a, c, g, t, i, -).
# 2) Builds bidirectional mappings between characters and integer token IDs.
# 3) Implements a sequence encoding function that converts a nucleotide string
#    into an int64 numpy array, mapping any unknown character to 'i'.


# we only recognize atcgi-
VOCAB = ['a','c','g','t','i','-']

# Character-to-index mapping, e.g. 'a'->0, 'c'->1, ...
'''
Will generate such a dictionary
{
  'a': 0,
  'c': 1,
  'g': 2,
  't': 3,
  'i': 4,
  '-': 5
}

'''
CHAR2ID = {c:i for i,c in enumerate(VOCAB)}

# Index-to-character mapping (inverse of CHAR2ID)
# Opposite process of the upper code
# Will generate this: {0:'a', 1:'c', 2:'g', 3:'t', 4:'i', 5:'-'}
ID2CHAR = {i:c for c,i in CHAR2ID.items()}

def encode_seq(s: str) -> np.ndarray:
    """
    Encode a nucleotide sequence string into an array of integer token IDs.

    - Input characters are lowercased.
    - Any character not in the predefined vocabulary is mapped to 'i'.
    - Output dtype is int64, which is compatible with PyTorch embedding layers.
    """
    s = str(s).lower()
    # Here we set all unknown ch into 'i'.
    unk = CHAR2ID['i']
    # [CHAR2ID.get(ch, unk) for ch in s means get all ch ins, if it is not in CHAR2ID,set it as unk (i)
    # pytorch default use int64
    return np.array([CHAR2ID.get(ch, unk) for ch in s], dtype=np.int64)


In [5]:
# === Build Spike k-mer training dataset with gap-only supervision ===
# This cell defines:
# 1) overlay_gaps_on_truth(): constructs an input scaffold x_spike by copying the gap markers
#    ('-' / 'i') from the voted scaffold onto an ungapped truth Spike sequence.
# 2) SpikeGapKmerDataset: generates (x_window, y_window) training pairs via sliding windows:
#       - x_window: truth_spike with gaps overlaid at voted-scaffold gap sites
#       - y_window: original truth_spike (ground truth)
#    Supervision is applied ONLY at positions where x_window has a gap marker (LOSS_ON),
#    and other positions are masked with -100 so they are ignored by CrossEntropyLoss.
# 3) collate_batch(): stacks samples into a batch dict compatible with HuggingFace-style models.



def overlay_gaps_on_truth(truth_spike: str) -> str:
    """
    Copy gap markers from the voted scaffold onto a truth Spike sequence.

    Input:
      truth_spike: ungapped (or fully specified) Spike region sequence (length L)
    Output:
      x_spike: same length as truth_spike, but positions in G_idx are replaced with
              vote_scaffold_spike[i] (typically '-' or 'i'), creating the model input scaffold.
    """
    x = list(truth_spike)
    for i in G_idx:
        x[i] = vote_scaffold_spike[i]  # keep scaffold marker ('i' or '-')
    return "".join(x)

class SpikeGapKmerDataset(Dataset):
    """
    Create k-mer training samples from many truth sequences using sliding windows.

    For each sequence:
      - truth_spike = Sequence[SPIKE_SLICE]
      - x_spike = overlay_gaps_on_truth(truth_spike)
      - generate (x_window, y_window) pairs by sliding a window of length k with given stride
      - keep only windows that contain at least one gap marker (LOSS_ON) in x_window

    Labels:
      - labels[t] = y_window[t] ONLY if x_window[t] is a gap marker (LOSS_ON)
      - labels[t] = -100 otherwise (ignored by PyTorch cross-entropy loss)
    """
    def __init__(self, df: pd.DataFrame, k=256, stride=128, max_windows_per_seq=None, seed=42):
        # --- Hyperparameters for k-mer window sampling ---
        # k: window length (number of tokens/bases per training example)
        # stride: step size between consecutive windows (overlap if stride < k)

        # Make sure k/stride is int to prevent it from error
        self.k = int(k)
        self.stride = int(stride)

        rng = random.Random(seed)

        # Store all training samples as pairs of strings: (x_window, y_window)
        # x_window: scaffold-like input (truth with gap markers overlaid)
        # y_window: ground-truth target (original truth spike window)
        self.samples = []

        # Iterate over each sequence row in the dataframe
        for _, row in df.iterrows():
            # Full genome (or full-length sequence) from the dataframe
            truth_full = row["Sequence"]
            # Extract Spike region by slicing with SPIKE_SLICE
            truth_spike = truth_full[SPIKE_SLICE]
            # Length sanity check: must match voted scaffold spike length L
            if len(truth_spike) != L:
                continue

            # Build model input scaffold for this sequence:
            # copy the voted scaffold gap markers ('-'/'i') into the truth spike
            x_spike = overlay_gaps_on_truth(truth_spike)


            # --- Sliding window generation over Spike region ---
            # We slide from start=0 to start=(L-k) inclusive, stepping by `stride`.
            # Each window produces:
            #   xw: input window from x_spike (contains gaps at some positions)
            #   yw: target window from truth_spike (pure ATCG ground truth)
            # We KEEP ONLY windows that contain at least one supervised position
            # (i.e., at least one gap marker in xw), otherwise the loss would be all masked.
            windows = []
            for start in range(0, L - self.k + 1, self.stride):
                # Input window (scaffold-like)
                xw = x_spike[start:start+self.k]
                # Filter: skip windows with no gap markers (no training signal)
                if not any(ch in LOSS_ON for ch in xw):
                    continue
                # Target window (ground truth)
                yw = truth_spike[start:start+self.k]
                # Save this (input, target) pair for later encoding in __getitem__
                windows.append((xw, yw))

            # Optional: cap the number of windows contributed by each sequence
            # This helps control dataset size and avoid a few sequences dominating training.
            if max_windows_per_seq is not None and len(windows) > max_windows_per_seq:
                windows = rng.sample(windows, max_windows_per_seq)

            # Add all windows from this sequence into the global sample list
            self.samples.extend(windows)

        print(f"[SpikeGapKmerDataset] k={self.k} stride={self.stride} windows={len(self.samples)}")

    # count number of samples
    def __len__(self):
        return len(self.samples)

    # get 1 sample from samples
    def __getitem__(self, idx):
        """
        Return one training example for the DataLoader.

        Each sample was precomputed in __init__ as a tuple of strings:
        (xw, yw)
            xw: input window from x_spike (truth spike with scaffold gap markers overlaid)
            yw: target window from truth_spike (ground truth bases)

        Output:
        input_ids: LongTensor of shape [k] (token IDs for xw)
        labels:    LongTensor of shape [k]
                - labels[t] = target token ID at position t IF xw[t] is a gap marker (LOSS_ON)
                - labels[t] = -100 otherwise (ignored by CrossEntropyLoss)
        """

        # Fetch the (input_window, target_window) pair (both are length-k strings)
        xw, yw = self.samples[idx]

        # Encode the input window into integer token IDs (int64/long for PyTorch embeddings)
        input_ids = torch.tensor(encode_seq(xw), dtype=torch.long)

        # Encode the target (ground truth) window into integer token IDs (numpy array)
        y_ids = encode_seq(yw)

        #here we construct the skeleton of labels, first we set all value=-100
        #(-100 means this position isn't a gap, we don't need to predict it)
        # This means: positions that are NOT gaps will not contribute to the loss/gradients.
        labels = np.full_like(y_ids, fill_value=-100)

        # Unmask only the supervised positions: where the input has a gap marker
        # (LOSS_ON typically includes '-' and/or 'i').
        for t, ch in enumerate(xw):
            if ch in LOSS_ON:
                labels[t] = y_ids[t]        # the correct base the model should predict at this gap

        # Convert labels to LongTensor for the model
        labels = torch.tensor(labels, dtype=torch.long)

        return input_ids, labels

# Here we combine several samples of __getitem__ and make it become a batch

"""
{
"input_ids": tensor(shape=(16,k))
"labels": tensor(shape=(16,k))
}
"""

def collate_batch(batch):
    """
    Collate function for DataLoader.
    Stacks a list of (input_ids, labels) into a batch dict:
      {
        "input_ids": Tensor[batch_size, k],
        "labels":    Tensor[batch_size, k]
      }
    """
    # collect all input ids
    # b:is an element in a batch
    # So this is equal with
    '''
    [
      input_ids_0,   # (k,)
      input_ids_1,   # (k,)
      ...
      input_ids_15   # (k,)
    ]
    '''
    # Finally we have things like
    '''
    [
      [token_0, token_1, ..., token_k-1],   # sample 0
      [token_0, token_1, ..., token_k-1],   # sample 1
      ...
      [token_0, token_1, ..., token_k-1],   # sample 15
    ]
    '''
    input_ids = torch.stack([b[0] for b in batch], dim=0)


    # collect all labels
    # Similar with collecting ids
    labels    = torch.stack([b[1] for b in batch], dim=0)
    return {"input_ids": input_ids, "labels": labels}


In [6]:
# === Build DataLoader and inspect one training batch ===
# This cell:
# 1) Defines global hyperparameters for k-mer training.
# 2) Instantiates the SpikeGapKmerDataset (runs __init__ once).
# 3) Wraps the dataset with a PyTorch DataLoader.
#    - DataLoader will automatically call __len__ and __getitem__ as needed.
# 4) Manually pulls one batch to verify tensor shapes and
#    the number of supervised (non-masked) tokens.

K = 256
STRIDE = 128
BATCH = 16

# Instantiate the dataset.
# This executes SpikeGapKmerDataset.__init__ ONCE and precomputes all sliding-window samples.
train_ds = SpikeGapKmerDataset(df_train, k=K, stride=STRIDE, max_windows_per_seq=None)


# Create DataLoader.
# DataLoader does NOT load data immediately.
# Instead, it will call:
#   - train_ds.__len__() to know dataset size
#   - train_ds.__getitem__(idx) to fetch samples on demand
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, collate_fn=collate_batch, drop_last=True)

# Create a batch iterator from the DataLoader
# iter(train_loader) returns an iterator object
# next(...) triggers fetching ONE batch:
#   - DataLoader selects BATCH indices
#   - calls __getitem__ for each index
#   - merges them using collate_batch
batch = next(iter(train_loader))

# Inspect batch tensor shapes
print(batch["input_ids"].shape, batch["labels"].shape)

# Count how many tokens in this batch are actually supervised
# (labels != -100 are the positions contributing to the loss)
print("num supervised tokens in batch:", int((batch["labels"] != -100).sum()))


[SpikeGapKmerDataset] k=256 stride=128 windows=57575
torch.Size([16, 256]) torch.Size([16, 256])
num supervised tokens in batch: 796


In [ ]:
# === Train Cell B7: Minimal GPT-2 style causal Transformer for char-level modeling ===



import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# In this class, we make sure we predict each position from its left side. For example, we predict t from 0....t-1, and t+1... cannot be involved.
# In this module, we implement multi-head causal self-attention.
# "Causal" means that each position t is only allowed to attend to positions <= t,
# and is strictly forbidden from attending to future positions (t+1, t+2, ...).
# This enforces an autoregressive (left-to-right) prediction constraint,
# which is essential for GPT-style language modeling.
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd: int, n_head: int, dropout: float):
        super().__init__()


        # multiple head attention will evenly divide embedding dimensions into heads, so it must be divisible
        # The embedding dimension must be divisible by the number of heads,
        # because we split the embedding evenly across attention heads.
        # Here, our n_embd=128, n_head=4 --> head_dim=32
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head

        # A single linear layer that projects the input embeddings
        # into concatenated Query, Key, and Value vectors.
        # Input:  (B, T, n_embd)
        # Output: (B, T, 3 * n_embd)
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=True)

        # Output projection that mixes information from all heads
        # back into the original embedding dimension.
        self.proj = nn.Linear(n_embd, n_embd, bias=True)


        # Dropout applied to attention weights and output projection
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x):
        # x shape: (B, T, C)
        # B = batch size (here is 16)
        # T = sequence length (here is K=256T)
        # C = embedding dimension (n_embd)(Here is 128)
        B, T, C = x.shape


        # Here, we generate q,k,v
        # self.qkv is a matrix, the future q,k,v now is a layer of self.qkv.
        # Project input embeddings into Q, K, V
        # qkv shape: (B, T, 3C)
        qkv = self.qkv(x)                      # (B, T, 3C)

        # Split the last dimension into q, k, v
        # Each has shape: (B, T, C)
        q, k, v = qkv.split(C, dim=2)          # each: (B, T, C)

        # Reshape for multi-head attention:
        # (B, T, C) -> (B, T, n_head, head_dim) -> (B, n_head, T, head_dim)
        # Each head attends independently over the sequence.
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        # attention scores: (B, nh, T, T)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        # causal mask: prevent attending to future tokens
        # mask shape: (1, 1, T, T)
        causal = torch.tril(torch.ones(T, T, device=x.device, dtype=torch.bool)).view(1, 1, T, T)
        att = att.masked_fill(~causal, float("-inf"))

        # Normalize attention scores into probabilities
        att = F.softmax(att, dim=-1)
        att = self.attn_drop(att)

        # Weighted sum of values
        # Result shape: (B, n_head, T, head_dim)
        y = att @ v                            # (B, nh, T, hs)

        # Concatenate heads back together:
        # (B, n_head, T, head_dim) -> (B, T, C)
        y = y.transpose(1, 2).contiguous().view(B, T, C)  # (B, T, C)

        # Final projection and dropout
        y = self.resid_drop(self.proj(y))      # (B, T, C)
        return y

#MLP is necessary after attention
# The MLP (feed-forward network) inside a Transformer block.
# In GPT-style Transformers, each block consists of:
#   LayerNorm -> Self-Attention -> Residual
#   LayerNorm -> MLP (FFN)      -> Residual
#
# This MLP is applied *independently at each time step* (position-wise),
# i.e., it does NOT mix information across different positions (attention does that).
# Its role is to provide non-linearity and increase model capacity by expanding
# the embedding dimension (typically 4x) and then projecting back.
class MLP(nn.Module):
    def __init__(self, n_embd: int, dropout: float):
        super().__init__()

        # First projection expands the channel dimension (C -> 4C),
        # increasing representational capacity.
        self.fc1 = nn.Linear(n_embd, 4 * n_embd)

        # Second projection compresses back to original embedding size (4C -> C).
        self.fc2 = nn.Linear(4 * n_embd, n_embd)

        # Dropout for regularization (applied after the second linear layer).
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        # x shape: (B, T, C)
        # The linear layers operate on the last dimension (C),
        # so the transformation is position-wise across (B, T).
        x = self.fc1(x)     # (B, T, C) -> (B, T, 4C)
        x = F.gelu(x)       # non-linear activation (GPT commonly uses GELU)
        x = self.fc2(x)     # (B, T, 4C) -> (B, T, C)
        x = self.drop(x)    # regularization
        return x


# A single Transformer block (GPT-style, Pre-LayerNorm).
#
# Structure:
#   x = x + SelfAttention(LayerNorm(x))
#   x = x + MLP(LayerNorm(x))
#
# Key points:
# - Residual connections ("x + ...") help optimization and allow training deeper networks.
# - Pre-LN (LayerNorm before sublayer) is a common GPT-style design and is often
#   more stable to train than Post-LN for deeper models.
# - Attention mixes information across positions; MLP is position-wise and adds nonlinearity.# LayerNorm before MLP (Pre-LN)
class Block(nn.Module):
    def __init__(self, n_embd: int, n_head: int, dropout: float):
        super().__init__()

        # LayerNorm before attention (Pre-LN)
        self.ln1 = nn.LayerNorm(n_embd)

        # Multi-head causal self-attention sublayer
        self.attn = CausalSelfAttention(n_embd, n_head, dropout)

        # LayerNorm before MLP (Pre-LN)
        self.ln2 = nn.LayerNorm(n_embd)

        # Position-wise feed-forward network (MLP/FFN)
        self.mlp = MLP(n_embd, dropout)


    def forward(self, x):
        # x shape: (B, T, C)
        # Residual path 1: attention
        x = x + self.attn(self.ln1(x))

        # Residual path 2: MLP
        x = x + self.mlp(self.ln2(x))
        return x


class GPT2TokenPredictor(nn.Module):
    """
    Minimal GPT-2 style causal Transformer language model for char-level tokens.

    Inputs/Outputs:
      - input_ids: LongTensor of shape (B, T) containing token IDs.
      - logits: FloatTensor of shape (B, T, vocab_size), the unnormalized scores
               for each token at each position.
      - If labels is provided, compute next-token (autoregressive) cross-entropy loss
        with ignore_index=-100, using the standard LM shift:
            predict token at position t+1 using context up to position t
            shift_logits = logits[:, :-1, :]
            shift_labels = labels[:, 1:]
    """
    def __init__(
        self,
        vocab_size: int,
        # Here we define the most length the model deal with at a time, because our k-mer window is 256 length long, we set it as k-mer length directly.
        max_len: int,
        n_layer: int = 4,
        n_head: int = 4,
        # embedding ： what is the dimensionality of the vector used to represent each token
        n_embd: int = 128,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.max_len = max_len

        # Token embedding: maps token IDs -> vectors (B, T) -> (B, T, C)
        # Here we change all token id into a 128-dimensional vector
        self.tok_emb = nn.Embedding(vocab_size, n_embd)

        # Position embedding: maps positions [0..max_len-1] -> vectors (1, T) -> (1, T, C)
        # broadcasted over batch and added to token embeddings
        # Here we give each position in sequence a vector, make the model know the position is the nth position.
        self.pos_emb = nn.Embedding(max_len, n_embd)


        # Here, a portion of the vector elements are randomly set to zero, preventing the model form overfitting
        self.drop = nn.Dropout(dropout)

        # Here we change the vector sequence into a contextual representation
        # Transformer blocks: stack of GPT-style (Pre-LN) blocks
        self.blocks = nn.ModuleList([Block(n_embd, n_head, dropout) for _ in range(n_layer)])

        # Final LayerNorm
        self.ln_f = nn.LayerNorm(n_embd)

        #language model head
        # LM head: projects hidden states back to vocabulary logits
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)

        # Weight tying: share token embedding weights with the output projection
        # (common GPT trick that reduces parameters and can improve generalization)
        # tie weights (optional but common)
        self.lm_head.weight = self.tok_emb.weight

    def forward(self, input_ids, labels=None):
        # input_ids: (B, T)
        # B = batch size T = length of k-mer window
        B, T = input_ids.shape

        # Safety check: position embedding table is only defined up to max_len
        if T > self.max_len:
            raise ValueError(f"Sequence length T={T} exceeds max_len={self.max_len}. Set max_len>=k.")

        # Positions [0..T-1], shaped as (1, T) so it broadcasts across batch
        #.unssueeze（0）: Here we add one more dimension, make(0,T) into (1, T), then it will become (B, T) sometime
        pos = torch.arange(0, T, device=input_ids.device).unsqueeze(0)  # (1, T)

        # Input embedding: token + position
        x = self.tok_emb(input_ids) + self.pos_emb(pos)  # (B, T, C)
        x = self.drop(x)

        # Contextualize representations with stacked Transformer blocks
        # make the model understand the contextual repaltionships
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)

        # very important
        # For every sequence in batch (B), for every position in it (T), we output a length = vocab_size(6) vector
        # x: want is x? it means for bth sequence, at tth position, the model get a C dimensional-vector after contextural thinking
        # lm_head(x): what is it? it is a matrix(vocab_size x n_embd), here vocab_size = 6 (atcgi-), n_embd = 128, so it is (6,128)
        # more clearly, lm_head(x) put the 128-dimensional vector into 6 categories, and value those dimensions based on those categories.
        # overall, we input a x(b,t), a 128-dimensional vector, and output a 6-dimensional (atcgi-) vector, then we got the score based on 6-dimensional vector
        logits = self.lm_head(x)  # (B, T, vocab_size)

        # labels: comes from Dataset __getitem__, it is a sequence with same length with input_ids
        # most positions of labels is -100, which means we will not calculate loss on this position(which is the function of the code below)
        if labels is None:
            # return a simple object-like dict
            return {"logits": logits}

        # Standard LM shift:
        # predict token t+1 given tokens <= t
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()

        # calcualte loss
        loss = F.cross_entropy(
            shift_logits.view(-1, self.vocab_size),
            shift_labels.view(-1),
            ignore_index=-100
        )

        return type("LMOutput", (object,), {"loss": loss, "logits": logits})()


# ---- instantiate model (uses VOCAB from your earlier cell) ----
model = GPT2TokenPredictor(
    vocab_size=len(VOCAB),
    max_len=K,        # IMPORTANT: set to your k-mer length
    n_layer=4,
    n_head=4,
    n_embd=128,
    dropout=0.1
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
print("Model params:", sum(p.numel() for p in model.parameters())/1e6, "M")


In [ ]:
# LayerNorm before MLP (Pre-LN)

In [8]:
# batch = next(iter(train_loader))
# input_ids = batch["input_ids"].to(device)
# labels = batch["labels"].to(device)
#
# out = model(input_ids=input_ids, labels=labels)
# print("loss:", float(out.loss.item()))
# print("logits shape:", out.logits.shape)
# print("supervised tokens (labels!=-100):", int((labels != -100).sum()))


In [9]:
# === Training loop: optimize the GPT-style model on gap-supervised batches ===
# This cell:
# 1) Switches the model to training mode (enables dropout, etc.).
# 2) Runs multiple epochs over the DataLoader.
# 3) For each batch:
#    - moves input_ids/labels to the target device (GPU/CPU)
#    - runs forward pass with labels to compute loss
#    - backpropagates and updates parameters using AdamW
# 4) Reports average loss per epoch.
#
# Note: labels use -100 as ignore_index, so the loss is computed ONLY on supervised
# gap positions (after the model's LM shift logic).

model.train()
epochs = 3

# Iterate over batches produced by DataLoader
# DataLoader will call Dataset.__getitem__ and collate_batch under the hood.
for ep in range(1, epochs+1):
    total_loss = 0.0
    steps = 0
    for batch in train_loader:

        # Move batch tensors to the same device as the model
        input_ids = batch["input_ids"].to(device)
        labels    = batch["labels"].to(device)

        # Clear accumulated gradients from the previous step
        optimizer.zero_grad()

        # Forward pass:
        # - returns logits and (if labels is provided) a cross-entropy loss
        out = model(input_ids=input_ids, labels=labels)
        # Support both object-like output (.loss) and raw loss return
        loss = out.loss if hasattr(out, "loss") else out

        # Backprop: compute gradients w.r.t. model parameters
        loss.backward()

        # Update model parameters (AdamW step)
        optimizer.step()

        # Track average loss for reporting
        total_loss += float(loss.item())
        steps += 1

    print(f"epoch {ep}: loss={total_loss/steps:.4f}")


epoch 1: loss=0.8154
epoch 2: loss=0.3722
epoch 3: loss=0.3511


In [10]:
# === Load evaluation scaffold + truth spike from exported files ===

SPIKE_LEN = 3822  # optional sanity

with open("eval_scaffold_spike_new.txt", "r") as f:
    scaffold_spike_new = f.readline().strip().lower()

with open("chosen_truth_spike.txt", "r") as f:
    chosen_truth_spike = f.readline().strip().lower()

print("loaded scaffold_spike_new len:", len(scaffold_spike_new))
print("loaded chosen_truth_spike len:", len(chosen_truth_spike))

assert len(scaffold_spike_new) == len(chosen_truth_spike), "length mismatch"
if SPIKE_LEN is not None:
    assert len(scaffold_spike_new) == SPIKE_LEN, "unexpected spike length (check slice coords)"

# gap indices for evaluation
GAP_CHARS = set(["i", "-"])
G_idx = [i for i,ch in enumerate(scaffold_spike_new) if ch in GAP_CHARS]
print("gap sites in eval scaffold:", len(G_idx))


loaded scaffold_spike_new len: 3822
loaded chosen_truth_spike len: 3822
gap sites in eval scaffold: 199


In [11]:
# === Inference Cell (5 truths): sliding windows + probability fusion, loop over eval_manifest_5truths.csv ===
# === Inference (5 truths): sliding windows + probability fusion over gap sites ===
# This cell evaluates the trained GPT-style model on 5 held-out truth sequences.
#
# Main idea:
# - We have a scaffold spike sequence containing gap markers ('-' / 'i').
# - We slide a window (length k, stride STRIDE) over the scaffold.
# - For each window, we run the causal LM once to get logits for all positions.
# - For each gap position t inside the window, we use logits[t-1] to predict token at t
#   (standard causal LM: position t is predicted from context up to t-1).
# - We convert logits[t-1] to probabilities, keep only ATCG probs, renormalize,
#   and fuse predictions from multiple overlapping windows using a center-weighted average.
# - Finally, we fill gap positions with argmax of fused ATCG probabilities and compute gap-only accuracy.


import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

# Switch to evaluation mode (disables dropout etc.)
model.eval()

# ---- load manifest produced by C3-scaffoldDataset_prep.ipynb ----
# The manifest is expected to contain columns like:
#   ID, truth_file, scaffold_file
manifest = pd.read_csv("eval_manifest_5truths.csv")
print("Loaded manifest rows:", len(manifest))
display(manifest)

# Define which characters are treated as gaps in scaffold
GAP_CHARS = set(["i","-"])

# Define allowed target alphabet for evaluation/filling (ATCG only)
ATCG = ['a','c','g','t']
ATCG_SET = set("acgt")

# Precompute token IDs for ATCG for efficient indexing on GPU
ATCG_IDS = torch.tensor([CHAR2ID[c] for c in ATCG], dtype=torch.long, device=device)

# Sliding window parameters (should match training window setup)
k = K
stride = STRIDE

def center_weight(t, k):
    """
    Compute a center-based weight for position t within a window of length k.

    Intuition:
      - Predictions near the center of a sliding window are usually more reliable
        than predictions near the edges (because window boundaries limit context).
      - We therefore fuse overlapping-window predictions using higher weights
        for center positions and lower weights for edge positions.

    This implements a simple linear "tent" weight:
      weight = 1 at the center, decreasing linearly to ~0 at the edges.

    This def means: if a position is predicted by 2 different windows, which window will be used?
    Answer:  The one whose center more near with the position will contribute more
    """

    # Center index in continuous coordinates:
    # for k=256, center = 127.5
    center = (k - 1) / 2.0

    # Normalized distance from center in [0, 1] (approximately)
    d = abs(t - center) / center if center > 0 else 0.0

    # Convert distance to weight: closer to center -> larger weight
    return 1.0 - d


def fill_one_scaffold(scaf: str):
    """
    Fill a single scaffold spike sequence at gap positions using fused window predictions.

    Returns:
      filled_spike: scaffold string with predicted ATCG bases inserted at gap sites
      gap_positions: list of indices where the scaffold has gap markers ('-'/'i')
      unfilled_positions: gap indices that received no usable prediction (computed later)
    """

    # Normalize input scaffold string: lowercase and remove surrounding whitespace/newlines
    scaf = str(scaf).lower().strip()
    # Sequence length (Spike region length)
    L = len(scaf)

    # Identify all gap positions in the scaffold (indices to fill)
    gap_pos = [i for i,ch in enumerate(scaf) if ch in GAP_CHARS]


    # Accumulators for probability fusion across overlapping windows:
    # probs_sum[pos] stores the (weighted) sum of predicted probabilities over ATCG at absolute position pos
    # shape: (L, 4) corresponding to [A, C, G, T]
    # What is accumulator? Because each position can be predicted by different sliding windows, so w need to merge their predictions

    probs_sum = np.zeros((L, 4), dtype=np.float64)

    # weight_sum[pos] stores the total accumulated weight contributed by all windows covering position pos
    # shape: (L,)
    weight_sum = np.zeros(L, dtype=np.float64)

    # Disable gradient tracking during inference to save memory and speed up computation
    with torch.no_grad():

        # Slide a window of length k across the scaffold sequence
        # start ranges over absolute positions in the full scaffold: [0, stride, 2*stride, ...]
        # last start is L-k (inclusive) so that xw has length k.
        for start in range(0, L - k + 1, stride):

            # Extract the local window (substring) from the scaffold
            xw = scaf[start:start+k]
            # Skip windows that contain no gaps at all (no positions to fill -> no need to run model)
            if not any(ch in GAP_CHARS for ch in xw):
                continue


            # Encode window into token IDs and add batch dimension:
            # (k,) -> (1, k)
            # encode_seq(xw), transfer the str int a k lenth token id array
            # unsqueeze, (k,) -> (1,k) because our model forward needs inputs_ids is (B,T)
            input_ids = torch.tensor(encode_seq(xw), dtype=torch.long, device=device).unsqueeze(0)  # (1,k)

            # Forward pass through the model (no labels -> returns logits only)
            out = model(input_ids=input_ids)

            #Support both dict output {"logits": ...} and object output .logits
            logits = out["logits"] if isinstance(out, dict) else out.logits

            # Remove batch dimension: (1, k, vocab) -> (k, vocab)
            logits = logits[0]  # (k,vocab)

            # Iterate over positions within the window
            for t, ch in enumerate(xw):

                # Only attempt to fill positions that are gaps in the scaffold window
                if ch not in GAP_CHARS:
                    continue

                # For causal LM, token at position t is predicted from logits at position (t-1).
                # If t == 0, there is no previous position inside the window, so skip.
                if t == 0:
                    continue

                # Convert logits at (t-1) to a probability distribution over the full vocab
                p = F.softmax(logits[t-1], dim=-1)

                # Keep only probabilities for {A,C,G,T} and renormalize to sum to 1.
                # This forces the filler to output only valid bases even if model assigns mass to 'i' or '-'.
                p_atcg = p.index_select(0, ATCG_IDS)
                p_atcg = p_atcg / (p_atcg.sum() + 1e-12)

                # Weight this window's contribution based on how central the position is
                w = center_weight(t, k)

                # Convert local window index t to absolute position in the full sequence
                abs_pos = start + t

                # Accumulate weighted probabilities into numpy buffers (fusion across windows)
                probs_sum[abs_pos] += (w * p_atcg).detach().cpu().numpy()
                weight_sum[abs_pos] += w

    # ---- Decode/fill the scaffold using fused probabilities ----
    filled = list(scaf)
    unfilled = []

    for pos in gap_pos:
        # If a gap position never received any prediction (no covering window or always t==0),
        # leave it unfilled and record it.
        if weight_sum[pos] <= 0:
            unfilled.append(pos)
            continue

        # Weighted average probability over windows, then choose the most likely base
        pred_idx = int(np.argmax(probs_sum[pos] / weight_sum[pos]))
        filled[pos] = ATCG[pred_idx]

    return "".join(filled), gap_pos, unfilled

# Containers for per-sequence evaluation results
results = []
filled_outputs = {}  # optional: store filled sequence per ID in memory

# ---- Evaluate each held-out case listed in the manifest ----
for _, r in manifest.iterrows():

    # Sequence identifier (for reporting / saving)
    seq_id = str(r["ID"])

    # Load truth and scaffold sequences from files
    # .strip() removes trailing newlines; .lower() normalizes to lowercase
    truth = open(r["truth_file"]).read().strip().lower()
    scaf  = open(r["scaffold_file"]).read().strip().lower()

    # Sanity check: the scaffold and truth must align position-by-position
    assert len(truth) == len(scaf), f"Length mismatch for {seq_id}"

    # Fill gap positions in the scaffold using the model + window fusion
    filled_spike, gap_pos, unfilled = fill_one_scaffold(scaf)

    # Evaluate only on valid gap sites:
    # - positions that are gaps in the scaffold (gap_pos)
    # - truth at that position is a valid base (ATCG)
    # - prediction at that position is also a valid base (ATCG)
    eval_sites = [pos for pos in gap_pos if truth[pos] in ATCG_SET and filled_spike[pos] in ATCG_SET]

    # Count correct predictions at evaluated gap sites
    correct = sum(filled_spike[pos] == truth[pos] for pos in eval_sites)

    # Gap-only accuracy (avoid division by zero if eval_sites is empty)
    acc = correct / max(len(eval_sites), 1)

    # Store per-sequence metrics
    results.append({
        "ID": seq_id,
        "gap_sites": len(gap_pos),
        "eval_sites": len(eval_sites),
        "unfilled": len(unfilled),
        "correct": correct,
        "gap_only_acc": acc
    })

    # Optionally keep the filled output sequence for later inspection/saving
    filled_outputs[seq_id] = filled_spike

# Build a results dataframe, ranked by gap-only accuracy
res_df = pd.DataFrame(results).sort_values("gap_only_acc", ascending=False)
display(res_df)

print("mean gap_only_acc:", float(res_df["gap_only_acc"].mean()))
print("std  gap_only_acc:", float(res_df["gap_only_acc"].std(ddof=1)))


Loaded manifest rows: 5


,ID,truth_file,scaffold_file,gap_count
0,EPI_ISL_11935954,truth_spike__EPI_ISL_11935954.txt,eval_scaffold_spike__EPI_ISL_11935954.txt,199
1,EPI_ISL_12957585,truth_spike__EPI_ISL_12957585.txt,eval_scaffold_spike__EPI_ISL_12957585.txt,199
2,EPI_ISL_10094228,truth_spike__EPI_ISL_10094228.txt,eval_scaffold_spike__EPI_ISL_10094228.txt,199
3,EPI_ISL_12162837,truth_spike__EPI_ISL_12162837.txt,eval_scaffold_spike__EPI_ISL_12162837.txt,199
4,EPI_ISL_9494671,truth_spike__EPI_ISL_9494671.txt,eval_scaffold_spike__EPI_ISL_9494671.txt,199


,ID,gap_sites,eval_sites,unfilled,correct,gap_only_acc
1,EPI_ISL_12957585,199,199,0,199,1.000000
2,EPI_ISL_10094228,199,199,0,198,0.994975
4,EPI_ISL_9494671,199,199,0,198,0.994975
0,EPI_ISL_11935954,199,199,0,197,0.989950
3,EPI_ISL_12162837,199,199,0,197,0.989950


mean gap_only_acc: 0.9939698492462312
std  gap_only_acc: 0.004204321741377282


In [12]:
res_df.to_csv("eval_results_5truths.csv", index=False)
print("saved eval_results_5truths.csv")

res_df["errors"] = res_df["gap_sites"] - res_df["correct"]
display(res_df[["ID","gap_sites","correct","errors","gap_only_acc"]])
print("mean errors per seq:", float(res_df["errors"].mean()))

print("Leak check:", set(CHOSEN_IDS).intersection(set(df_train["ID"].astype(str))))


saved eval_results_5truths.csv


,ID,gap_sites,correct,errors,gap_only_acc
1,EPI_ISL_12957585,199,199,0,1.000000
2,EPI_ISL_10094228,199,198,1,0.994975
4,EPI_ISL_9494671,199,198,1,0.994975
0,EPI_ISL_11935954,199,197,2,0.989950
3,EPI_ISL_12162837,199,197,2,0.989950


mean errors per seq: 1.2
Leak check: set()


In [13]:
# === Detailed per-truth inspection ===

for _, r in manifest.iterrows():
    seq_id = r["ID"]
    truth = open(r["truth_file"]).read().strip().lower()
    scaf  = open(r["scaffold_file"]).read().strip().lower()
    pred  = filled_outputs[seq_id]

    GAP_CHARS = set(["i","-"])
    gap_pos = [i for i,ch in enumerate(scaf) if ch in GAP_CHARS]

    errors = [i for i in gap_pos if pred[i] != truth[i]]

    print("\n" + "="*80)
    print(f"ID: {seq_id}")
    print(f"Gap sites: {len(gap_pos)} | Errors: {len(errors)}")

    if errors:
        print("First few error positions (0-based in spike):", errors[:10])
        print("Examples (pos : truth → pred):")
        for pos in errors[:5]:
            print(f"  {pos}: {truth[pos]} → {pred[pos]}")
    else:
        print("No errors 🎉")

    print("-"*80)
    print("Scaffold (with gaps):")
    print(scaf)
    print("-"*80)
    print("Prediction:")
    print(pred)
    print("-"*80)
    print("Truth:")
    print(truth)



ID: EPI_ISL_11935954
Gap sites: 199 | Errors: 2
First few error positions (0-based in spike): [1250, 1335]
Examples (pos : truth → pred):
  1250: g → t
  1335: g → a
--------------------------------------------------------------------------------
Scaffold (with gaps):
atgtttgtttttcttgttttattgccactagtctctagtcagtgtgttaatcttacaaccagaactcaat---------catacactaattctttcacacgtggtgtttattaccctgacaaagttttcagatcctcagttttacattcaactcaggacttgttcttacctttcttttccaatgttacttggttccatgcta------tctctgggaccaatggtactaagaggtttgataaccctgtcctaccatttaatgatggtgtttattttgcttccactgagaagtctaacataataagaggctggatttttggtactactttagattcgaagacccagtccctacttattgttaataacgctactaatgttgttattaaagtctgtgaatttcaattttgtaatgatccatttttgg---------accacaaaaacaacaaaagttggatggaaagtgagttcagagtttattctagtgcgaataattgcacttttgaatatgtctctcagccttttcttatggaccttgaaggaaaacagggtaatttcaaaaatcttagggaatttgtgtttaagaatattgatggttattttaaaatatattctaagcacacgcctatta---tagggcgtgatctccctcagggtttttcggctttagaaccattggtagatttgccaataggtattaacatcactaggtttcaaactttacttgctt

In [14]:
import pandas as pd
import numpy as np

manifest = pd.read_csv("eval_manifest_5truths.csv")
GAP_CHARS = set(["i", "-"])

# Sanity: ensure we have predictions in memory
assert "filled_outputs" in globals(), "Need `filled_outputs` dict from your inference cell (ID -> filled_spike)."

rows = []

for _, r in manifest.iterrows():
    seq_id = str(r["ID"])
    truth = open(r["truth_file"]).read().strip().lower()
    scaf  = open(r["scaffold_file"]).read().strip().lower()
    pred  = filled_outputs[seq_id].strip().lower()

    assert len(truth) == len(scaf) == len(pred), f"Length mismatch for {seq_id}"
    L = len(truth)

    # Position-wise adjacency: compare adjacent pairs at each position
    truth_pairs = [truth[i:i+2] for i in range(L-1)]
    pred_pairs  = [pred[i:i+2]  for i in range(L-1)]

    adj_match = np.array([1 if truth_pairs[i] == pred_pairs[i] else 0 for i in range(L-1)], dtype=np.int32)
    adj_acc_all = float(adj_match.mean())

    # Gap positions in scaffold
    gap_pos = set(i for i, ch in enumerate(scaf) if ch in GAP_CHARS)

    # Edges adjacent to gaps: edge i is (i, i+1)
    gap_edges = [i for i in range(L-1) if (i in gap_pos) or (i+1 in gap_pos)]
    if len(gap_edges) > 0:
        adj_acc_gap_edges = float(adj_match[gap_edges].mean())
    else:
        adj_acc_gap_edges = float("nan")

    rows.append({
        "ID": seq_id,
        "L": L,
        "gap_sites": len(gap_pos),
        "edges_total": L-1,
        "edges_adjacent_to_gaps": len(gap_edges),
        "adj_acc_all": adj_acc_all,
        "adj_acc_gap_edges": adj_acc_gap_edges,
        "adj_errors_all": int((L-1) - adj_match.sum()),
        "adj_errors_gap_edges": int(len(gap_edges) - adj_match[gap_edges].sum()) if len(gap_edges) > 0 else 0,
    })

adj_df = pd.DataFrame(rows).sort_values("adj_acc_gap_edges", ascending=True)
display(adj_df)

print("=== Summary across 5 truths ===")
print("mean adj_acc_all:", float(adj_df["adj_acc_all"].mean()))
print("std  adj_acc_all:", float(adj_df["adj_acc_all"].std(ddof=1)))

print("mean adj_acc_gap_edges:", float(adj_df["adj_acc_gap_edges"].mean()))
print("std  adj_acc_gap_edges:", float(adj_df["adj_acc_gap_edges"].std(ddof=1)))


,ID,L,gap_sites,edges_total,edges_adjacent_to_gaps,adj_acc_all,adj_acc_gap_edges,adj_errors_all,adj_errors_gap_edges
0,EPI_ISL_11935954,3822,199,3821,207,0.998953,0.980676,4,4
3,EPI_ISL_12162837,3822,199,3821,207,0.998953,0.980676,4,4
2,EPI_ISL_10094228,3822,199,3821,207,0.999477,0.990338,2,2
4,EPI_ISL_9494671,3822,199,3821,207,0.999477,0.990338,2,2
1,EPI_ISL_12957585,3822,199,3821,207,1.000000,1.000000,0,0


=== Summary across 5 truths ===
mean adj_acc_all: 0.9993718921748235
std  adj_acc_all: 0.0004379272580654554
mean adj_acc_gap_edges: 0.9884057971014493
std  adj_acc_gap_edges: 0.008083671753952403
